In [1]:
import torch
from collections import defaultdict

In [33]:
def load_data(file_pattern, num_files):
    data_by_coords = defaultdict(list)

    for i in range(1, num_files + 1):
        filename = file_pattern.format(i)
        data = torch.load(filename)
        
        for entry in data:
            coords = tuple(entry['coordinates'])
            answer = entry['answer'].squeeze(0)  # Reshape [1, 8, 6] to [8, 6]
            data_by_coords[coords].append(answer)

    return data_by_coords


In [34]:
data_by_coords = load_data("/mnt/d/sources/cgan/playground/dataset/3p6_time_{}.torch", 10)

In [35]:
len(data_by_coords[-117, -76, -25])

10

In [36]:
len(data_by_coords)

33598

In [43]:
len(data_by_coords[-117, -76, -25])

10

In [37]:
data_by_coords[-117, -76, -25][0]

tensor([[ 0.0208, -0.0564,  0.0217,  0.0407, -0.0383,  0.0788],
        [-0.0520,  0.1376, -0.3669, -0.0146, -0.2297, -0.0339],
        [ 0.1281,  0.0225, -0.1117, -0.1245, -0.0141,  0.0845],
        [-0.0250, -0.0405,  0.1901, -0.0901,  0.1176,  0.0058],
        [ 0.0156,  0.0457,  0.2644, -0.7112, -0.0264, -0.0290],
        [ 0.0317,  0.0128,  0.1344,  0.0088, -0.0224,  0.0215],
        [ 0.1371,  0.1382,  0.0763,  0.1292,  0.0388,  0.0030],
        [-0.1457,  0.1424, -0.0142, -0.0054,  0.1672,  0.0336]],
       grad_fn=<SqueezeBackward1>)

In [38]:
# 2. Organize data into source-target pairs
def create_source_target(data_by_coords, source_length=4, target_length=1):
    pairs = [] 
    for coords, tensor_list in data_by_coords.items():
        for i in range(0, len(tensor_list) - source_length - target_length + 1, source_length + target_length):
            source = tensor_list[i: i + source_length]
            target = tensor_list[i + source_length: i + source_length + target_length]
            pairs.append((source, target))
    
    return pairs

In [39]:
data_pairs = create_source_target(data_by_coords)

In [40]:
type(data_pairs)

list

In [41]:
len(data_pairs)

67196

In [44]:
for i in range(0, 10 - 4 - 1 + 1, 4 + 1):
    print(i)

0
5


In [42]:
data_pairs[0]

([tensor([[ 0.0208, -0.0564,  0.0217,  0.0407, -0.0383,  0.0788],
          [-0.0520,  0.1376, -0.3669, -0.0146, -0.2297, -0.0339],
          [ 0.1281,  0.0225, -0.1117, -0.1245, -0.0141,  0.0845],
          [-0.0250, -0.0405,  0.1901, -0.0901,  0.1176,  0.0058],
          [ 0.0156,  0.0457,  0.2644, -0.7112, -0.0264, -0.0290],
          [ 0.0317,  0.0128,  0.1344,  0.0088, -0.0224,  0.0215],
          [ 0.1371,  0.1382,  0.0763,  0.1292,  0.0388,  0.0030],
          [-0.1457,  0.1424, -0.0142, -0.0054,  0.1672,  0.0336]],
         grad_fn=<SqueezeBackward1>),
  tensor([[ 0.0208, -0.0564,  0.0217,  0.0407, -0.0383,  0.0788],
          [-0.0520,  0.1376, -0.3669, -0.0146, -0.2297, -0.0339],
          [ 0.1281,  0.0225, -0.1117, -0.1245, -0.0141,  0.0845],
          [-0.0250, -0.0405,  0.1901, -0.0901,  0.1176,  0.0058],
          [ 0.0156,  0.0457,  0.2644, -0.7112, -0.0264, -0.0290],
          [ 0.0317,  0.0128,  0.1344,  0.0088, -0.0224,  0.0215],
          [ 0.1371,  0.1382,  0.0763,

In [45]:
from torch.utils.data import DataLoader, Dataset

class Seq2SeqDataset(Dataset):
    def __init__(self, data_pairs):
        self.data_pairs = data_pairs

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        return self.data_pairs[idx]

dataset = Seq2SeqDataset(data_pairs)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)



In [46]:
import torch.nn as nn

class Seq2SeqTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, d_model, nhead, num_encoder_layers, num_decoder_layers, dropout):
        super(Seq2SeqTransformer, self).__init__()

        self.transformer = nn.Transformer(d_model, nhead, num_encoder_layers, num_decoder_layers, dropout=dropout)
        self.source_embedding = nn.Linear(input_dim, d_model)
        self.target_embedding = nn.Linear(output_dim, d_model)
        self.output_layer = nn.Linear(d_model, output_dim)

    def forward(self, src, tgt):
        src = self.source_embedding(src)
        tgt = self.target_embedding(tgt)
        output = self.transformer(src, tgt)
        return self.output_layer(output)




In [ ]:
# Hyperparameters
INPUT_DIM = 6   # Source tensor dimension
OUTPUT_DIM = 6  # Target tensor dimension
D_MODEL = 512
NHEAD = 8
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
DROPOUT = 0.1

model = Seq2SeqTransformer(INPUT_DIM, OUTPUT_DIM, D_MODEL, NHEAD, NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, DROPOUT)

In [53]:
import torch.optim as optim

# Define the loss function and the optimizer
loss_function = nn.MSELoss()  # Mean squared error loss, suitable for regression tasks
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training hyperparameters
EPOCHS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)

for epoch in range(EPOCHS):
    total_loss = 0.0

    for batch in dataloader:
        src_list, tgt_list = batch
        print(len(src_list))
        print(src_list[0])
        
        src = torch.stack(src_list).to(DEVICE)
        tgt = torch.stack(tgt_list).to(DEVICE)

        print(src.shape)
        print(tgt.shape)

        optimizer.zero_grad()

        # Since we don't have a separate tgt_input for teacher forcing, 
        # we can use zeros of the same shape as tgt as a placeholder.
        tgt_input = torch.zeros_like(tgt).to(DEVICE)

        # Forward pass
        output = model(src, tgt_input)

        # Compute the loss
        loss = loss_function(output, tgt)

        # Backward pass
        loss.backward()

        # Optimization step
        optimizer.step()

        total_loss += loss.item()

    # Print average loss for the epoch
    print(f"Epoch {epoch + 1}/{EPOCHS} - Loss: {total_loss / len(dataloader)}")


4
tensor([[[ 4.3570e-03, -4.7891e-02,  1.3581e-02,  3.5427e-02, -3.3866e-02,
           7.3402e-02],
         [-5.3677e-02,  1.2846e-01, -3.4814e-01, -5.8559e-02, -2.1094e-01,
          -4.7344e-02],
         [ 1.3809e-01,  2.9408e-02, -1.2235e-01, -1.2356e-01, -1.2915e-02,
           7.5639e-02],
         ...,
         [ 2.8547e-02, -3.3189e-05,  1.2365e-01, -1.4776e-03, -1.5687e-02,
           2.8802e-02],
         [ 1.2657e-01,  1.1264e-01, -6.8625e-03,  1.2685e-01,  3.7878e-02,
           1.5485e-02],
         [-1.4260e-01,  8.6674e-02, -2.5854e-02, -8.1599e-04,  1.3712e-01,
           3.0158e-02]],

        [[-7.0297e-03, -4.8004e-02,  1.8859e-02,  4.5817e-02, -3.1417e-02,
           7.0298e-02],
         [-4.7363e-02,  1.2578e-01, -3.6919e-01, -5.9711e-02, -2.2849e-01,
          -2.7640e-02],
         [ 1.5007e-01,  1.9842e-02, -1.1705e-01, -1.1241e-01, -2.2000e-02,
           7.8207e-02],
         ...,
         [ 3.1679e-02,  1.0621e-02,  1.2794e-01,  8.3089e-03, -2.2444e-02,
  

AssertionError: query should be unbatched 2D or batched 3D tensor but received 4-D query tensor

In [8]:
# Specify the path to the saved new .torch file
# new_file_path = "dataset/3p6_time_1.torch"
new_file_path = "/mnt/d/sources/cgan/playground/dataset/3p6_time_1.torch"

In [4]:
# Load the new data from file
loaded_new_data = torch.load(new_file_path)

In [5]:
# Check if there are at least 5 items in the loaded data
if len(loaded_new_data) < 5:
    print("There are less than 5 items in the loaded data.")
else:
    # Print out the first 5 items
    for i in range(1):
        print("Item", i + 1)
        print("Coordinates:", loaded_new_data[i]['coordinates'])
        print("Velocity:", loaded_new_data[i]['velocity'])
        print("Answer:", loaded_new_data[i]['answer'])
        print('---' * 20)


Item 1
Coordinates: [-117, -76, -25]
Velocity: tensor([[[0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322,
          0.5347, 0.5347, 0.5322, 0.5322, 0.5347, 0.5347, 0.5366, 0.5322,
          0.5322, 0.5347, 0.5347, 0.5366, 0.5322, 0.5322, 0.5347, 0.5347,
          0.5366, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322,
          0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347,
          0.5322, 0.5322, 0.5347, 0.5347, 0.5347, 0.5322, 0.5322, 0.5347,
          0.5347, 0.5366, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322,
          0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5347,
          0.5347, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322,
          0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5322, 0.5347,
          0.5322, 0.5322, 0.5322, 0.5322, 0.5347, 0.5322, 0.5322, 0.5322,
          0.5322, 0.5347, 0.5322, 0.5322, 0.5322, 0.5322, 0.5347, 0.5322,
          0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5347, 0.5347,

In [6]:
len(loaded_new_data)

33598

In [7]:
loaded_new_data.dtype

AttributeError: 'list' object has no attribute 'dtype'

In [8]:
type(loaded_new_data)

list

In [9]:
loaded_new_data[0]

{'coordinates': [-117, -76, -25],
 'velocity': tensor([[[0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322,
           0.5347, 0.5347, 0.5322, 0.5322, 0.5347, 0.5347, 0.5366, 0.5322,
           0.5322, 0.5347, 0.5347, 0.5366, 0.5322, 0.5322, 0.5347, 0.5347,
           0.5366, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322,
           0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347,
           0.5322, 0.5322, 0.5347, 0.5347, 0.5347, 0.5322, 0.5322, 0.5347,
           0.5347, 0.5366, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322,
           0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5347,
           0.5347, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322,
           0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5322, 0.5347,
           0.5322, 0.5322, 0.5322, 0.5322, 0.5347, 0.5322, 0.5322, 0.5322,
           0.5322, 0.5347, 0.5322, 0.5322, 0.5322, 0.5322, 0.5347, 0.5322,
           0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5